# Revise an existing FDA indication using the tools only

This notebook demonstrates the update workflow for `NDA208716` using only the public `moalmanac-fda-curation` command-line tools. It does not invoke the curation skill or use an agent to make review decisions.

In the accompanying database checkout, `NDA208716` is `doc:fda:verzenio`, represented by the March 2023 label. At the time this notebook was created, Drugs@FDA reports a newer approved label. The preflight gate verifies both facts at runtime rather than assuming they remain true.

## 1. Configure the run

Use a fresh work directory for each revision evaluation. Record the curation-tool and `moalmanac-db` commits alongside any completed run because both the database and FDA label history can change. The commands verify the `moalmanac-db` checkout and name any missing input.

In [ ]:
from pathlib import Path
from IPython.display import FileLink, display
import json
import subprocess
import sys

APPLICATION_NUMBER = "NDA208716"
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent
DATABASE_DIR = REPO_ROOT.parent / "moalmanac-db"
# Both notebooks operate on the same FDA-label update session.
WORK_DIR = REPO_ROOT / "outputs" / "tool-only-new-nda208716"

print(f"Application: {APPLICATION_NUMBER}")
print(f"Database: {DATABASE_DIR}")
print(f"Work directory: {WORK_DIR}")

In [ ]:
def run_tool(*arguments):
    command = [sys.executable, "-m", "moalmanac_fda_curation.cli", *map(str, arguments)]
    print("$", " ".join(command))
    return subprocess.run(command, cwd=REPO_ROOT, check=True, text=True)

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def show_file(path):
    path = Path(path).resolve()
    if not path.is_file():
        raise FileNotFoundError(path)
    display(FileLink(str(path)))

def record_decision(stage, decision, indication_index, overrides=None, keep_existing_fields=None, note=None):
    arguments = [
        "record-decision", "--work-dir", WORK_DIR,
        "--stage", stage, "--decision", decision,
        "--indication-index", indication_index,
    ]
    for field, value in (overrides or {}).items():
        arguments.extend(["--override", f"{field}={json.dumps(value)}"])
    for field in keep_existing_fields or []:
        arguments.extend(["--keep-existing-field", field])
    if note:
        arguments.extend(["--note", note])
    return run_tool(*arguments)

## 2. Check setup and confirm the revision route

A real revision requires both an existing MOAlmanac document and a newer approved FDA label. The notebook stops if either condition is absent.

In [ ]:
run_tool("check-setup")
status_path = WORK_DIR / "intermediate" / "curation-status.json"
if not status_path.exists():
    run_tool(
        "check-curation-status",
        "--application-number", APPLICATION_NUMBER,
        "--database-dir", DATABASE_DIR,
        "--output-json", status_path,
    )
status = load_json(status_path)
display(status)
if not status.get("previously_curated"):
    raise RuntimeError("NDA208716 is not yet curated in this database checkout; run the new-curation notebook first.")
if not status.get("newer_label_available"):
    raise RuntimeError("NDA208716 is curated, but no newer approved FDA label is available.")

## 3. Extract the latest label and reconcile indications

`find-new-indications` prepares the latest document proposal, extracts latest-label indications, maps them against existing MOAlmanac indications, and writes review files for new, `not_found`, and `uncertain` results. A `not_found` result does not establish that FDA removed the indication; it may reflect extraction or matching behavior.

In [ ]:
matches_path = WORK_DIR / "intermediate" / "indication-matches.json"
if not matches_path.exists():
    run_tool(
        "find-new-indications",
        "--application-number", APPLICATION_NUMBER,
        "--database-dir", DATABASE_DIR,
        "--work-dir", WORK_DIR,
    )
matches = load_json(matches_path)
for mapping in matches.get("mappings", []):
    print(
        mapping.get("classification"),
        mapping.get("existing_indication_id"),
        mapping.get("latest_indication_index"),
        mapping.get("reason"),
        sep=" | ",
    )
for path in sorted((WORK_DIR / "review").rglob("*.md")):
    show_file(path)

### Resolve reconciliation exceptions

Review every `not_found` or `uncertain` mapping before continuing. The current CLI generates evidence packets for these exceptions but does not provide a command that rewrites an incorrect reconciliation result. If a mapping is wrong, stop and correct or regenerate the reconciliation artifact through the supported tool workflow before revision screening.

In [ ]:
mapping_exceptions = [
    item for item in matches.get("mappings", [])
    if item.get("classification") in {"not_found", "uncertain"}
]
CONFIRM_MAPPING_EXCEPTIONS_REVIEWED = False
if mapping_exceptions and not CONFIRM_MAPPING_EXCEPTIONS_REVIEWED:
    raise RuntimeError("Review and resolve every not_found or uncertain mapping before continuing.")

## 4. Reuse new-indication decisions from the companion notebook

New indications were already reviewed in `curate-new-indication-tool-only.ipynb`. Both notebooks use the same work directory, so those decisions are reused here. This notebook verifies that every new candidate has an indication decision and does not ask you to curate index 3 again.

In [ ]:
new_candidates = matches.get("new_indication_candidates", [])
NEW_CANDIDATE_INDEXES = [item["latest_indication_index"] for item in new_candidates]
decisions_path = WORK_DIR / "review" / "decisions.json"
if NEW_CANDIDATE_INDEXES and not decisions_path.is_file():
    raise RuntimeError("Complete the new-indication notebook before continuing.")
decisions = load_json(decisions_path) if decisions_path.is_file() else {"indications": {}}
missing_new_decisions = [
    index for index in NEW_CANDIDATE_INDEXES
    if not decisions.get("indications", {}).get(str(index), {}).get("indication", {}).get("decision")
]
if missing_new_decisions:
    raise RuntimeError(
        f"Complete indication review for new indexes {missing_new_decisions} in the new-indication notebook."
    )
print(f"Reusing completed new-indication decisions for indexes: {NEW_CANDIDATE_INDEXES}")

In [ ]:
print("No new-indication review is repeated in this notebook.")

In [ ]:
print("Continue to revision screening below.")

## 5. Identify changed matched indications

The tool compares the baseline and latest Indications and Usage sections deterministically, asks one batched LLM call to associate changes with existing indications, and creates screening packets for indications classified as `revised`. Revision detection is intentionally sensitive; the curator decides whether a detected change warrants replacing the MOAlmanac indication.

In [ ]:
targets_path = WORK_DIR / "intermediate" / "revision-targets.json"
if not targets_path.exists():
    run_tool("find-revised-indications", "--database-dir", DATABASE_DIR, "--work-dir", WORK_DIR)
targets = load_json(targets_path)
for target in targets.get("targets", []):
    print(
        target["latest_indication_index"],
        target["existing_indication_id"],
        target.get("label_change_ids"),
        target.get("reason"),
        sep=" | ",
    )
for path in sorted((WORK_DIR / "review" / "revision-screening").glob("*.md")):
    show_file(path)

## 6. Screen each possible revision

Choose `use_latest` when the change is meaningful enough to replace the existing indication; assembly then suggests a new indication record built from the latest fields. Choose `keep_existing` when the detected label change does not warrant a replacement. Overrides may accompany `use_latest`.

In [ ]:
# CURATOR GATE: one entry per target: index -> (decision, overrides).
# Each detected change is wording-only and does not alter the population, therapy,
# disease, or eligibility criteria, so retain all four existing records.
REVISION_SCREENING_DECISIONS = {
    0: ("keep_existing", {}),  # adult patients -> adults
    1: ("keep_existing", {}),  # wording fix and biomarker abbreviations
    2: ("keep_existing", {}),  # adults wording and biomarker abbreviations
    4: ("keep_existing", {}),  # adult patients -> adults
}
CONFIRM_REVISION_SCREENING = True
target_indexes = {item["latest_indication_index"] for item in targets.get("targets", [])}
if not target_indexes:
    CONFIRM_REVISION_SCREENING = True
if not CONFIRM_REVISION_SCREENING:
    raise RuntimeError("Review every revision-screening packet before confirming.")
if set(REVISION_SCREENING_DECISIONS) != target_indexes:
    raise ValueError("Provide one screening decision for every revision target.")
for index, (decision, overrides) in REVISION_SCREENING_DECISIONS.items():
    record_decision("revision", decision, index, overrides=overrides)

## 7. Prepare detailed reviews for accepted revisions

Only targets screened as `use_latest` proceed. The description review shows the existing MOAlmanac statement description alongside the new proposal. Approval matching is restricted to label events after the previously curated label; the curator may instead keep the existing approval date for the new record.

In [ ]:
run_tool("prepare-revision-reviews", "--work-dir", WORK_DIR)
ACCEPTED_REVISION_INDEXES = sorted(
    index for index, (decision, _) in REVISION_SCREENING_DECISIONS.items()
    if decision == "use_latest"
)
for path in sorted((WORK_DIR / "review").rglob("*.md")):
    show_file(path)

In [ ]:
# CURATOR GATE: add description and approval decisions for every accepted revision.
# To keep the existing approval date, use an edited approval decision with
# keep_existing_fields=["initial_approval_date"].
REVISION_DOWNSTREAM_DECISIONS = {}
# Example structure:
# {2: {
#     "description": {"decision": "accepted", "overrides": {}},
#     "approval": {"decision": "edited", "overrides": {},
#                  "keep_existing_fields": ["initial_approval_date"]},
# }}
CONFIRM_REVISION_DOWNSTREAM_DECISIONS = not ACCEPTED_REVISION_INDEXES
if not CONFIRM_REVISION_DOWNSTREAM_DECISIONS:
    raise RuntimeError("Review all accepted-revision description and approval packets.")
if set(REVISION_DOWNSTREAM_DECISIONS) != set(ACCEPTED_REVISION_INDEXES):
    raise ValueError("Provide downstream decisions for every accepted revision.")
for index, stages in REVISION_DOWNSTREAM_DECISIONS.items():
    for stage in ("description", "approval"):
        configuration = stages[stage]
        record_decision(
            stage, configuration["decision"], index,
            overrides=configuration.get("overrides"),
            keep_existing_fields=configuration.get("keep_existing_fields"),
        )

## 8. Assemble revision artifacts

Assembly suggests new indication records for accepted new indications and for changed indications the curator chose to replace, with `replacements.json` naming the existing indication each replacement replaces. Each new record reports the evergreen document and the dated document for its approval label; assembly reuses dated documents already in `moalmanac-db` and suggests the ones it lacks. It also writes new contribution records and document and URL patch artifacts, and leaves existing indication records unchanged. It validates every record against the `moalmanac-db` schemas and does not modify `moalmanac-db`.

In [ ]:
run_tool("assemble-revisions", "--work-dir", WORK_DIR, "--database-dir", DATABASE_DIR)
for path in sorted((WORK_DIR / "reviewed").glob("*.json")):
    print(f"\n{path.name}")
    display(load_json(path))
    show_file(path)

## Handoff

Apply the suggested records and patches to `moalmanac-db` in a separate, reviewed change. Preserve this work directory so the reconciliation, source diffs, review packets, decisions, and assembled records remain auditable.